# RSNA Knee Abnormality Detection — Image Baseline (skeleton)

This notebook is meant to run **on Kaggle**, not locally — the competition data
(247 GB of DICOM images) is mounted at `/kaggle/input/rsna-knee-abnormality-detection/`
on Kaggle's servers, and is not available on this machine.

Goal of this skeleton: get a full pipeline running end-to-end without errors —
load images, train a small CNN, predict, produce a valid `submission.csv` — as a
foundation to iterate on. It is **not** expected to score well yet:
- Only 58 of 4407 training studies currently have labels.
- We start from a single mid-sagittal slice per study (simplest possible input);
  extending to multi-slice / 3D / multi-plane fusion is the natural next step.

Constraints from the competition rules: <= 9h runtime, internet disabled at
submission time (so pretrained weights must come from Kaggle's local cache or a
Kaggle Dataset input, not a live download).

In [ ]:
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score

def pick_device() -> torch.device:
    if not torch.cuda.is_available():
        return torch.device("cpu")
    try:
        major, minor = torch.cuda.get_device_capability(0)
        cap_str = f"sm_{major}{minor}"
        if cap_str in torch.cuda.get_arch_list():
            return torch.device("cuda")
        print(f"GPU capability {cap_str} unsupported by this PyTorch build "
              f"(supports {torch.cuda.get_arch_list()}); falling back to CPU.")
    except Exception as e:
        print(f"GPU capability check failed ({e}); falling back to CPU.")
    return torch.device("cpu")


DEVICE = pick_device()
print("Device:", DEVICE)

In [ ]:
COMPETITION_SLUG = "rsna-knee-abnormality-detection"
DATA_DIR = Path(f"/kaggle/input/competitions/{COMPETITION_SLUG}")

TRAIN_CSV = DATA_DIR / "train.csv"
TRAIN_SERIES_CSV = DATA_DIR / "train_series.csv"
TEST_CSV = DATA_DIR / "test.csv"
TEST_SERIES_CSV = DATA_DIR / "test_series.csv"
TRAIN_SERIES_DIR = DATA_DIR / "train_series"
TEST_SERIES_DIR = DATA_DIR / "test_series"

LABELS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
    "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture",
]

IMG_SIZE = 224
RANDOM_STATE = 42
BATCH_SIZE = 8
N_EPOCHS = 5  # skeleton default — increase once the pipeline is validated

## Data loading

For each study we pick one series to keep things simple: prefer a **sagittal**
series (ACL/menisci are best seen sagittally), otherwise fall back to whatever
series is available. We then take the **middle slice** of that series as the
input image — the simplest possible baseline; multi-slice/3D is a follow-up.

In [ ]:
def load_labels():
    df = pd.read_csv(TRAIN_CSV)
    return df.dropna(subset=LABELS).reset_index(drop=True)


def pick_series(study_uid: str, series_df: pd.DataFrame) -> str | None:
    rows = series_df[series_df["StudyInstanceUID"] == study_uid]
    if rows.empty:
        return None
    sagittal = rows[rows["Anatomical_Plane"] == "Sagittal"]
    chosen = sagittal.iloc[0] if len(sagittal) else rows.iloc[0]
    return chosen["SeriesInstanceUID"]


def load_middle_slice(series_dir: Path) -> np.ndarray:
    dcm_files = sorted(series_dir.glob("*.dcm"))
    if not dcm_files:
        raise FileNotFoundError(f"No .dcm files in {series_dir}")

    def instance_number(path):
        try:
            return pydicom.dcmread(path, stop_before_pixels=True).InstanceNumber
        except Exception:
            return 0

    dcm_files = sorted(dcm_files, key=instance_number)
    mid = dcm_files[len(dcm_files) // 2]
    ds = pydicom.dcmread(mid)
    arr = ds.pixel_array.astype(np.float32)
    arr -= arr.min()
    if arr.max() > 0:
        arr /= arr.max()
    return arr  # shape (H, W), values in [0, 1]

In [ ]:
preprocess = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


class KneeMRIDataset(Dataset):
    def __init__(self, df: pd.DataFrame, series_df: pd.DataFrame, series_dir: Path, has_labels: bool):
        self.df = df.reset_index(drop=True)
        self.series_df = series_df
        self.series_dir = series_dir
        self.has_labels = has_labels

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        study_uid = row["StudyInstanceUID"]
        series_uid = pick_series(study_uid, self.series_df)
        img = load_middle_slice(self.series_dir / study_uid / series_uid)
        img_rgb = np.stack([img, img, img], axis=-1)  # fake 3-channel for pretrained CNN
        tensor = preprocess(img_rgb)

        if self.has_labels:
            y = torch.tensor(row[LABELS].values.astype(np.float32))
            return tensor, y
        return tensor, study_uid

## Model

A ResNet18 backbone with a 12-way sigmoid output head (multi-label). We try to
load ImageNet pretrained weights; if internet is disabled and they aren't
already cached in the Kaggle image, we fall back to random init so the
pipeline doesn't crash.

In [ ]:
def build_model(n_labels: int = len(LABELS)) -> nn.Module:
    try:
        backbone = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    except Exception as e:
        print(f"Could not load pretrained weights ({e}); using random init.")
        backbone = models.resnet18(weights=None)
    backbone.fc = nn.Linear(backbone.fc.in_features, n_labels)
    return backbone

## Training

Single train/val split (data is far too small for k-fold to be meaningful yet).
Reports per-label AUC and the macro average, same metric as the leaderboard.

In [ ]:
def train_one_fold(train_ds, val_ds):
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

    model = build_model().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.BCEWithLogitsLoss()

    for epoch in range(N_EPOCHS):
        model.train()
        total_loss = 0.0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * x.size(0)
        print(f"Epoch {epoch + 1}/{N_EPOCHS} — train loss: {total_loss / len(train_ds):.4f}")

    model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(DEVICE)
            probs = torch.sigmoid(model(x)).cpu().numpy()
            all_preds.append(probs)
            all_targets.append(y.numpy())
    return model, np.concatenate(all_preds), np.concatenate(all_targets)


def macro_auc(preds: np.ndarray, targets: np.ndarray) -> float:
    scores = []
    for i, label in enumerate(LABELS):
        if len(np.unique(targets[:, i])) < 2:
            continue
        scores.append(roc_auc_score(targets[:, i], preds[:, i]))
        print(f"  {label:<20} AUC={scores[-1]:.3f}")
    macro = float(np.mean(scores)) if scores else float("nan")
    print(f"Macro AUC: {macro:.3f}")
    return macro

In [ ]:
labeled_df = load_labels()
series_df = pd.read_csv(TRAIN_SERIES_CSV)
print(f"Labeled training studies: {len(labeled_df)}")

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
train_idx, val_idx = next(kf.split(labeled_df))
train_ds = KneeMRIDataset(labeled_df.iloc[train_idx], series_df, TRAIN_SERIES_DIR, has_labels=True)
val_ds = KneeMRIDataset(labeled_df.iloc[val_idx], series_df, TRAIN_SERIES_DIR, has_labels=True)

model, val_preds, val_targets = train_one_fold(train_ds, val_ds)
macro_auc(val_preds, val_targets)

## Inference on the test set

Builds `submission.csv` in the exact format expected by the leaderboard.

In [ ]:
test_df = pd.read_csv(TEST_CSV)
test_series_df = pd.read_csv(TEST_SERIES_CSV)
test_ds = KneeMRIDataset(test_df, test_series_df, TEST_SERIES_DIR, has_labels=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

model.eval()
uids, preds = [], []
with torch.no_grad():
    for x, study_uid in test_loader:
        x = x.to(DEVICE)
        probs = torch.sigmoid(model(x)).cpu().numpy()
        uids.extend(study_uid)
        preds.append(probs)

preds = np.concatenate(preds)
submission = pd.DataFrame(preds, columns=LABELS)
submission.insert(0, "StudyInstanceUID", uids)
submission.to_csv("submission.csv", index=False)
submission.head()